# Document Classification Pipeline - Full End-to-End Verification

This notebook demonstrates the complete lifecycle of a document within the Enterprise Knowledge Base (EKB):
1. **Step 0: Ingestion**: Uploading a local PDF with custom metadata to the Landing Zone.
2. **Step 1: DLP Security Gate**: Deterministic scanning and automated masking (for high-risk content).
3. **Step 2: Contextual Classification**: Multimodal Gemini 2.5 Flash reasoning over the (masked) document.
4. **Step 3: Routing & Persistence**: Moving original/masked files to domain buckets and indexing metadata in BigQuery.

In [1]:
import sys
import os
from loguru import logger
from google.cloud import storage, bigquery

# Ensure repo root is in path
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from pipelines.enterprise_knowledge_base import ClassificationPipeline
from pipelines.enterprise_knowledge_base.document_classification.config import EKB_CONFIG
from pipelines.enterprise_knowledge_base.document_classification.gcs_service import GCSService

print(f"Project ID: {EKB_CONFIG.PROJECT_ID}")

Project ID: p-dev-gce-60pf


## Step 0: Ingestion (Local File to Landing Zone)
We will upload a local PDF and attach the custom metadata expected by the pipeline.

In [2]:
def ingest_local_file(local_path: str, metadata: dict) -> str:
    """Simulates the Agent ADK Skill: Uploads a local file with metadata to GCS.
    
    This helper function is self-contained to avoid modifying core service classes.
    
    Args:
        local_path: Path to the local file.
        metadata: Dictionary of metadata (project, domain, trust-level, etc.)
        
    Returns:
        str: The landing zone GCS URI.
    """
    client = storage.Client()
    bucket_name = "enterprise_knowledgebase_landing_zone"
    filename = os.path.basename(local_path)
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)
    
    # Attach custom metadata (x-goog-meta-*)
    blob.metadata = metadata
    
    print(f"Uploading {filename} to gs://{bucket_name}/{filename} with metadata: {metadata}")
    blob.upload_from_filename(local_path, content_type="application/pdf")
    
    return f"gs://{bucket_name}/{filename}"


In [5]:
# Metadata from Design.md
sample_metadata = {
    "project": "alpha-strategy-2026",
    "domain": "finance",
    "trust-level": "wip",
    "uploader": "emmanuel.amador@endava.com",
    "creator-name": "Emmanuel Amador"
}


local_pdf_path = "../../../PASAPORTE.pdf" 

if os.path.exists(local_pdf_path):
    landing_zone_uri = ingest_local_file(local_pdf_path, sample_metadata)
    print(f"Ingestion successful: {landing_zone_uri}")
else:
    print(f"ERROR: Local file {local_pdf_path} not found. Please provide a valid path.")

Uploading PASAPORTE.pdf to gs://enterprise_knowledgebase_landing_zone/PASAPORTE.pdf with metadata: {'project': 'alpha-strategy-2026', 'domain': 'finance', 'trust-level': 'wip', 'uploader': 'emmanuel.amador@endava.com', 'creator-name': 'Emmanuel Amador'}
Ingestion successful: gs://enterprise_knowledgebase_landing_zone/PASAPORTE.pdf


## Step 1: DLP Security Gate (Scan & Mask)
Deterministic phase to detect high-risk data and protect privacy.

In [8]:
pipeline = ClassificationPipeline()

print(f"Triggering DLP Phase for: {landing_zone_uri}")
dlp_result = pipeline.dlp_trigger(landing_zone_uri)


sanitized_gcs_uri = dlp_result.sanitized_gcs_uri
print(f"DLP Result URI: {sanitized_gcs_uri}")
print(f"Proposed Tier: {dlp_result.proposed_classification_tier}")

2026-04-22 22:09:24.715 | INFO     | pipelines.enterprise_knowledge_base.document_classification.pipeline:dlp_trigger:76 - Triggering DLP scan for: gs://enterprise_knowledgebase_landing_zone/PASAPORTE.pdf
2026-04-22 22:09:24.716 | INFO     | pipelines.enterprise_knowledge_base.document_classification.dlp_service:inspect_gcs_file:38 - Starting DLP scan for: gs://enterprise_knowledgebase_landing_zone/PASAPORTE.pdf
2026-04-22 22:09:24.717 | DEBUG    | pipelines.enterprise_knowledge_base.document_classification.dlp_service:_get_custom_keywords_config:209 - Building custom keywords config for inspection.


Triggering DLP Phase for: gs://enterprise_knowledgebase_landing_zone/PASAPORTE.pdf


2026-04-22 22:09:25.748 | INFO     | pipelines.enterprise_knowledge_base.document_classification.dlp_service:inspect_gcs_file:66 - DLP Job created: projects/p-dev-gce-60pf/locations/global/dlpJobs/i-4113616863175615896
2026-04-22 22:09:25.856 | INFO     | pipelines.enterprise_knowledge_base.document_classification.dlp_service:wait_for_job:104 - Waiting for DLP Job... (Current state: PENDING)
2026-04-22 22:09:31.022 | INFO     | pipelines.enterprise_knowledge_base.document_classification.dlp_service:wait_for_job:104 - Waiting for DLP Job... (Current state: PENDING)
2026-04-22 22:09:36.141 | INFO     | pipelines.enterprise_knowledge_base.document_classification.dlp_service:wait_for_job:104 - Waiting for DLP Job... (Current state: RUNNING)
2026-04-22 22:09:41.364 | INFO     | pipelines.enterprise_knowledge_base.document_classification.dlp_service:wait_for_job:104 - Waiting for DLP Job... (Current state: RUNNING)
2026-04-22 22:09:46.586 | INFO     | pipelines.enterprise_knowledge_base.docu

DLP Result URI: gs://enterprise_knowledgebase_landing_zone/PASAPORTE_masked.pdf
Proposed Tier: 5


## Step 2: Contextual Classification (Gemini 2.5 Flash)
Probabilistic phase to refine the tier and domain using multimodal reasoning.

In [7]:
print("Triggering Gemini Contextual Phase...")
# Use the metadata extracted from GCS for Phase 2
blob_meta = pipeline._get_blob_metadata(landing_zone_uri)

verdict = pipeline.contextual_classification(
    sanitized_url=dlp_result.sanitized_gcs_uri,
    proposed_classification_tier=dlp_result.proposed_classification_tier,
    proposed_domain=blob_meta.proposed_domain,
    trust_level=blob_meta.trust_level
)

print("\n--- Gemini Verdict ---")
print(f"Final Tier: {verdict.final_classification_tier}")
print(f"Confidence: {verdict.confidence}")
print(f"Validated Domain: {verdict.final_domain}")
print(f"Summary: {verdict.file_description}")

2026-04-22 22:08:32.891 | DEBUG    | pipelines.enterprise_knowledge_base.document_classification.pipeline:_get_blob_metadata:108 - Extracting metadata for: gs://enterprise_knowledgebase_landing_zone/PASAPORTE.pdf
2026-04-22 22:08:32.891 | INFO     | pipelines.enterprise_knowledge_base.document_classification.gcs_service:get_blob_metadata:31 - Extracting detailed GCS metadata for: gs://enterprise_knowledgebase_landing_zone/PASAPORTE.pdf
2026-04-22 22:08:32.892 | DEBUG    | pipelines.enterprise_knowledge_base.document_classification.gcs_service:_parse_uri:132 - Parsing GCS URI into dictionary: gs://enterprise_knowledgebase_landing_zone/PASAPORTE.pdf
2026-04-22 22:08:32.988 | INFO     | pipelines.enterprise_knowledge_base.document_classification.pipeline:contextual_classification:55 - Starting contextual classification for: gs://enterprise_knowledgebase_landing_zone/PASAPORTE_masked.pdf
2026-04-22 22:08:32.988 | INFO     | pipelines.enterprise_knowledge_base.document_classification.gcs_se

Triggering Gemini Contextual Phase...


2026-04-22 22:08:33.187 | INFO     | pipelines.enterprise_knowledge_base.document_classification.gemini_service:classify_document:41 - Classifying document via Gemini: gs://enterprise_knowledgebase_landing_zone/PASAPORTE_masked.pdf



--- Gemini Verdict ---
Final Tier: 5
Confidence: 0.98
Validated Domain: finance
Summary: This document is a Mexican passport containing Personally Identifiable Information (PII) such as the holder's name, surname, nationality, date and place of birth, passport number, issuing and expiry dates, and a photograph. It also includes the holder's signature and the issuing authority. This information is highly sensitive and critical for identity verification purposes.


## Step 3: Routing & BQ Persistence
Relocation to domain buckets and BigQuery indexing.

In [ ]:
print("Executing final routing and BQ persistence...")

final_original_uri = pipeline.file_routing(
    original_landing_uri=landing_zone_uri,
    sanitized_landing_uri=dlp_result.sanitized_gcs_uri,
    final_domain=verdict.final_domain,
    final_security_tier=verdict.final_classification_tier,
    project_name=blob_meta.project_name,
    uploader_email=blob_meta.uploader_email
)

# For BQ metadata record, we need to know where the masked file went (if it exists)
final_sanitized_uri = None
if dlp_result.sanitized_gcs_uri != landing_zone_uri:
    dest_base = final_original_uri.rsplit("/", 1)[0]
    masked_filename = dlp_result.sanitized_gcs_uri.split("/")[-1]
    final_sanitized_uri = f"{dest_base}/{masked_filename}"

pipeline.metadata_bq(
    final_original_uri=final_original_uri,
    final_sanitized_uri=final_sanitized_uri,
    llm_classification_dict=verdict.model_dump(),
    blob_metadata_dict=blob_meta.model_dump()
)

print("\n--- Pipeline Complete ---")
print(f"Original File Secured at: {final_original_uri}")
if final_sanitized_uri:
    print(f"Masked File Secured at: {final_sanitized_uri}")
print("Metadata indexed in BigQuery knowledge_base.documents_metadata.")